# Algoritmo FP-Growth

Notebook elaborado siguiendo la rúbrica de la asignatura (ver `../../rubrica.md`).

**Autor:** Grupo E
**Fecha:** 13/06/2026

## 1. Descripción

**FP-Growth** (*Frequent Pattern Growth*) es un algoritmo de **minería de reglas de asociación** y descubrimiento de **itemsets frecuentes** en bases de datos transaccionales. Fue propuesto por Han, Pei y Yin en 2000 como una alternativa eficiente para encontrar patrones frecuentes sin generar candidatos explícitos.

Su objetivo es encontrar relaciones del tipo:

> *Si un cliente compra pan y mantequilla, entonces es probable que también compre leche.*

Formalmente, dada una base de transacciones $D = \{T_1, T_2, \dots, T_n\}$ donde cada $T_i$ es un conjunto de ítems, FP-Growth busca todos los itemsets $X \subseteq I$ cuyo **soporte** supere un umbral mínimo, y a partir de ellos genera **reglas de asociación** $X \Rightarrow Y$ con métricas de calidad como confianza y lift.

La idea central del algoritmo es construir una estructura compacta llamada **FP-Tree** (*Frequent Pattern Tree*), que codifica la frecuencia de los ítems y conserva la información necesaria para extraer patrones frecuentes mediante **crecimiento recursivo de patrones**.

### Casos de uso típicos
- Análisis de la cesta de la compra (*market basket analysis*).
- Recomendación de productos.
- Detección de patrones en logs web.
- Detección de co-ocurrencias en datos médicos.
- Bioinformática y análisis de secuencias discretizadas.

## 2. Bibtex y Referencias

### BibTeX
```bibtex
@inproceedings{han2000fpgrowth,
  title     = {Mining frequent patterns without candidate generation},
  author    = {Han, Jiawei and Pei, Jian and Yin, Yiwen},
  booktitle = {Proceedings of the 2000 ACM SIGMOD international conference on Management of data},
  pages     = {1--12},
  year      = {2000}
}

@article{han2004mining,
  title     = {Mining frequent patterns without candidate generation: A frequent-pattern tree approach},
  author    = {Han, Jiawei and Pei, Jian and Yin, Yiwen and Mao, Runying},
  journal   = {Data Mining and Knowledge Discovery},
  volume    = {8},
  number    = {1},
  pages     = {53--87},
  year      = {2004}
}
```

### APA
- Han, J., Pei, J., & Yin, Y. (2000). *Mining frequent patterns without candidate generation*. In **Proceedings of the 2000 ACM SIGMOD International Conference on Management of Data** (pp. 1–12).
- Han, J., Pei, J., Yin, Y., & Mao, R. (2004). *Mining frequent patterns without candidate generation: A frequent-pattern tree approach*. **Data Mining and Knowledge Discovery, 8**(1), 53–87.

## 3. Tipo de Modelo

| Criterio | Clasificación |
|---|---|
| **Método de aprendizaje** | No supervisado |
| **Por parámetros** | No paramétrico |
| **Datos de aprendizaje** | Offline (batch) |
| **Resultado del entrenamiento** | FP-Tree + itemsets frecuentes + reglas de asociación |

Notas:
- **No supervisado** porque no requiere etiquetas.
- **No paramétrico** porque no ajusta una forma funcional fija del modelo.
- **Offline** porque necesita recorrer la base de transacciones para construir la estructura inicial.
- El resultado es una representación compacta de patrones frecuentes y sus reglas asociadas.

## 4. Algoritmo de Entrenamiento

FP-Growth construye una estructura compacta llamada **FP-Tree** y después extrae patrones frecuentes mediante crecimiento recursivo de patrones sin generar candidatos explícitos.

### Pseudocódigo
```text
Entrada: D (transacciones), min_sup (soporte mínimo)
Salida: L = conjunto de itemsets frecuentes

1. Escanear la base y contar frecuencia de cada ítem.
2. Eliminar ítems con soporte menor que min_sup.
3. Ordenar los ítems frecuentes por frecuencia descendente.
4. Construir la FP-Tree insertando cada transacción filtrada y ordenada.
5. Para cada ítem frecuente, construir su base de patrones condicional.
6. Construir FP-Trees condicionales de manera recursiva.
7. Extraer todos los itemsets frecuentes encontrados.
```

### Componentes principales
- **Header table:** mantiene referencia a todos los nodos de un mismo ítem.
- **FP-Tree:** árbol compacto que resume las transacciones.
- **Conditional pattern base:** conjunto de rutas que terminan en un ítem dado.
- **Conditional FP-Tree:** árbol derivado para extraer patrones asociados a un sufijo.

### Métricas clave

- **Soporte:** $\text{sup}(X) = \dfrac{|\{T \in D : X \subseteq T\}|}{|D|}$
- **Confianza:** $\text{conf}(X \Rightarrow Y) = \dfrac{\text{sup}(X \cup Y)}{\text{sup}(X)}$
- **Lift:** $\text{lift}(X \Rightarrow Y) = \dfrac{\text{conf}(X \Rightarrow Y)}{\text{sup}(Y)}$

## 5. Supuestos y Restricciones

- **Datos transaccionales categóricos:** los ítems deben ser discretos.
- **Representación binaria:** cada transacción se interpreta como presencia o ausencia de ítems.
- **Umbrales definidos por el usuario:** `min_support` y, si se generan reglas, `min_confidence`.
- **Dependencia del orden de inserción:** el orden de los ítems influye en la compactación del árbol.
- **Uso de memoria:** aunque evita candidatos explícitos, la FP-Tree puede crecer mucho en datos muy densos o con alta cardinalidad.
- **No modela tiempo ni orden secuencial** entre transacciones.
- **Posibles reglas redundantes:** conviene post-filtrar por lift, conviction u otras métricas.

## 6. Tests / Métricas de validación

FP-Growth se valida con las siguientes métricas sobre los itemsets y reglas descubiertas:

- **Soporte (support)** — frecuencia relativa del itemset.
- **Confianza (confidence)** — probabilidad condicional $P(Y \mid X)$.
- **Lift** — fuerza de la asociación respecto a la independencia.
- **Leverage** — diferencia entre frecuencia observada y esperada.
- **Conviction** — fortaleza de la implicación de la regla.

Además, para evaluar el algoritmo como procedimiento de minería:

- **Tiempo de ejecución** — importante en bases grandes.
- **Consumo de memoria** — depende del tamaño de la FP-Tree.
- **Número de patrones extraídos** — útil para analizar sensibilidad a los umbrales.

La calidad práctica se complementa con validación cualitativa y revisión del dominio de aplicación.

---
## 7. Implementación práctica

Usaremos `mlxtend` para construir itemsets frecuentes y reglas de asociación en un dataset de ejemplo.

### 7.1 Instalación e imports

In [1]:
# Si se ejecuta en Colab o un entorno sin las librerías, descomentar:
# !pip install mlxtend pandas

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

print('Librerías cargadas correctamente')

Librerías cargadas correctamente


### 7.2 Dataset de ejemplo — Cesta de la compra

Usaremos una pequeña base de transacciones de supermercado para ilustrar el algoritmo.

In [2]:
transacciones = [
    ['pan', 'leche', 'huevos'],
    ['pan', 'pañales', 'cerveza', 'huevos'],
    ['leche', 'pañales', 'cerveza', 'cola'],
    ['pan', 'leche', 'pañales', 'cerveza'],
    ['pan', 'leche', 'pañales', 'cola'],
    ['pan', 'leche'],
    ['pan', 'cerveza'],
    ['leche', 'pañales'],
    ['pan', 'leche', 'pañales'],
    ['cerveza', 'cola'],
]

print(f'Número de transacciones: {len(transacciones)}')
for i, t in enumerate(transacciones, 1):
    print(f'T{i:>2}: {t}')

Número de transacciones: 10
T 1: ['pan', 'leche', 'huevos']
T 2: ['pan', 'pañales', 'cerveza', 'huevos']
T 3: ['leche', 'pañales', 'cerveza', 'cola']
T 4: ['pan', 'leche', 'pañales', 'cerveza']
T 5: ['pan', 'leche', 'pañales', 'cola']
T 6: ['pan', 'leche']
T 7: ['pan', 'cerveza']
T 8: ['leche', 'pañales']
T 9: ['pan', 'leche', 'pañales']
T10: ['cerveza', 'cola']


### 7.3 Codificación one-hot

`mlxtend` necesita una matriz booleana donde cada columna es un ítem y cada fila una transacción.

In [3]:
te = TransactionEncoder()
te_array = te.fit(transacciones).transform(transacciones)
df = pd.DataFrame(te_array, columns=te.columns_)
df

,cerveza,cola,huevos,leche,pan,pañales
0,False,False,True,True,True,False
1,True,False,True,False,True,True
2,True,True,False,True,False,True
3,True,False,False,True,True,True
4,False,True,False,True,True,True
5,False,False,False,True,True,False
6,True,False,False,False,True,False
7,False,False,False,True,False,True
8,False,False,False,True,True,True
9,True,True,False,False,False,False


### 7.4 Itemsets frecuentes con FP-Growth

Buscamos itemsets con soporte mayor o igual a `0.3`.

In [4]:
min_sup = 0.3
frecuentes = fpgrowth(df, min_support=min_sup, use_colnames=True)
frecuentes = frecuentes.sort_values('support', ascending=False).reset_index(drop=True)
frecuentes

,support,itemsets
0,0.7,frozenset({pan})
1,0.7,frozenset({leche})
2,0.6,frozenset({pañales})
3,0.5,frozenset({cerveza})
4,0.5,"frozenset({leche, pan})"
5,0.5,"frozenset({leche, pañales})"
6,0.4,"frozenset({pan, pañales})"
7,0.3,frozenset({cola})
8,0.3,"frozenset({leche, pan, pañales})"
9,0.3,"frozenset({cerveza, pañales})"


### 7.5 Generación de reglas de asociación

A partir de los itemsets frecuentes, generamos reglas con confianza mínima de `0.6`.

In [5]:
reglas = association_rules(frecuentes, metric='confidence', min_threshold=0.6)
columnas = ['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage', 'conviction']
reglas = reglas[columnas].sort_values('lift', ascending=False).reset_index(drop=True)
reglas

,antecedents,consequents,support,confidence,lift,leverage,conviction
0,frozenset({pañales}),frozenset({leche}),0.5,0.833333,1.190476,0.08,1.80
1,frozenset({leche}),frozenset({pañales}),0.5,0.714286,1.190476,0.08,1.40
2,"frozenset({pan, pañales})",frozenset({leche}),0.3,0.750000,1.071429,0.02,1.20
3,frozenset({leche}),frozenset({pan}),0.5,0.714286,1.020408,0.01,1.05
4,frozenset({pan}),frozenset({leche}),0.5,0.714286,1.020408,0.01,1.05
5,"frozenset({leche, pan})",frozenset({pañales}),0.3,0.600000,1.000000,0.00,1.00
6,frozenset({cerveza}),frozenset({pañales}),0.3,0.600000,1.000000,0.00,1.00
7,frozenset({pañales}),frozenset({pan}),0.4,0.666667,0.952381,-0.02,0.90
8,"frozenset({leche, pañales})",frozenset({pan}),0.3,0.600000,0.857143,-0.05,0.75
9,frozenset({cerveza}),frozenset({pan}),0.3,0.600000,0.857143,-0.05,0.75


### 7.6 Interpretación de resultados

- Las reglas con **lift > 1** indican asociación positiva entre antecedente y consecuente.
- Los itemsets con soporte alto representan patrones frecuentes dentro del conjunto de transacciones.
- Ajustar `min_support` y `min_threshold` controla el equilibrio entre cantidad y calidad de reglas.

In [6]:
print('Top 5 reglas por LIFT:')
for _, r in reglas.head(5).iterrows():
    ant = ', '.join(sorted(r['antecedents']))
    con = ', '.join(sorted(r['consequents']))
    print(f'  {{{ant}}}  =>  {{{con}}}')
    print(f'     sup={r["support"]:.3f}  conf={r["confidence"]:.3f}  lift={r["lift"]:.3f}\n')

Top 5 reglas por LIFT:
  {pañales}  =>  {leche}
     sup=0.500  conf=0.833  lift=1.190

  {leche}  =>  {pañales}
     sup=0.500  conf=0.714  lift=1.190

  {pan, pañales}  =>  {leche}
     sup=0.300  conf=0.750  lift=1.071

  {leche}  =>  {pan}
     sup=0.500  conf=0.714  lift=1.020

  {pan}  =>  {leche}
     sup=0.500  conf=0.714  lift=1.020



### 7.7 Traducción de resultados

A continuación, una interpretación en español de los resultados obtenidos:

- Los patrones más frecuentes describen combinaciones de productos que aparecen con mayor repetición en las transacciones.
- Las reglas con mayor lift sugieren relaciones de compra más fuertes que las esperadas por azar.
- Si una regla presenta confianza alta, entonces el consecuente aparece con frecuencia cuando está presente el antecedente.
- En un contexto comercial, estos patrones pueden apoyar recomendaciones de productos, promociones cruzadas y análisis de la cesta de compra.

## 8. Conclusión

- FP-Growth es un algoritmo **no supervisado y no paramétrico** para descubrir itemsets frecuentes y reglas de asociación.
- Su principal aporte es la construcción de una **FP-Tree** compacta que evita generar candidatos explícitos.
- La extracción recursiva de patrones condicionales permite encontrar asociaciones de forma eficiente.
- El rendimiento depende de los umbrales elegidos y de la estructura de los datos.
- En aplicaciones reales, FP-Growth es útil para análisis de compras, recomendación y descubrimiento de patrones frecuentes.